Se carga la base de datos

Se trabaja con el módulo auxiliar process_data, el cual define una carga personalizada para estos ejemplos de ataques a modelos fedrados. Para más detalles sobre como efectuar la carga de datos con Flex ir a la documentación correspondiente.

Este módulo permite la carga de dataset de procesamiento de imágenes como: Mnist, Fmnist, Cifar10 y Cifar100. Además del dataset tabular nursery.

En este ejemplo se trbajará con el dataset nursery a diferencia de los anteriores

In [1]:
from process_data import *
from copy import deepcopy

flex_data, server_id = load_and_preprocess_horizontal_tabular(dataname="nursery", trasnform=False, nodes=2)
adv_data = flex_data[server_id]

A continuación, se define la arquitectura de los modelos locales de los clientes. Para el presente ejemplo se trabaja con modelos neuronales de pytorch.

Se utiliza el módulo networks_models, quien contiene una serie de modelos neuronales auxiliares de pytorch, para el trabajo con las bases de datos anteriormente mencionadas. Además se utiliza el módulo auxiliar networks_execution, que define la ejecución del entrenamiento y otros detalles de estos modelos.

Para establecer un modelo personalizado, ir a la documentación de Flex.

In [2]:
from networks_models import *
from networks_execution import *
from flex.pool import init_server_model
from flex.model import FlexModel

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

#net_config = ExecutionNetwork()
model_base = build_model_for_tabular(in_put = 8, inner_put = 15,out_put = 4, dataname='nursery')

@init_server_model
def build_server_model():
    server_flex_model = FlexModel()

    model = model_base

    server_flex_model["model"] = model.to(device)
    # Required to store this for later stages of the FL training process
    server_flex_model["criterion"] = nn.CrossEntropyLoss()
    server_flex_model["optimizer_func"] = optim.Adam(model.parameters(), lr=0.001)
    server_flex_model["optimizer_kwargs"] = {}

    return server_flex_model

Se define la arquitectura del modelo federado

In [3]:
from flex.pool import FlexPool
clients = 1

pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func = build_server_model
    )

#selected_test_clients_pool = pool.clients.select(clients)
#selected_test_clients = selected_test_clients_pool.clients

Se define la función para desplegar el modelo global en cada cliente

In [4]:
from flex.pool.decorators import (  # noqa: E402
    deploy_server_model,
)

@deploy_server_model
def deploy_serv(server_flex_model: FlexModel): 

    new_model = deepcopy(server_flex_model)

    return new_model

#pool.servers.map(deploy_serv, selected_test_clients)

Se define la ronda de entrenamiento local de un cliente. Para este ejemplo se utiliza el código definido para el entrenamiento de un modelo del framework adversarial robustness toolbox. Como forma de enlazar ammbas herramientas que se enfocan en el mismo objetivo

In [5]:
from art.estimators.classification import PyTorchClassifier

def train(client_flex_model: FlexModel, client_data: Dataset):

    x_train, y_train = transform_tabular(dataname="nursery", dataset = client_data)
    columns = x_train[0].shape
    min_value_for_attr=np.min(x_train, axis = 0)
    max_value_for_attr=np.max(x_train, axis = 0)
    train_dataset = transform_numpy_to_tensor_dataset(x_train, y_train)
    client_dataloader = DataLoader(train_dataset, batch_size = 64)
    model = client_flex_model["model"]
    model = model.to(device)
    client_flex_model["previous_model"] = deepcopy(
        model
    )

    optimizer = client_flex_model["optimizer_func"]
    criterion = client_flex_model["criterion"]

    classifier = PyTorchClassifier(
                model=model,
                clip_values=(min_value_for_attr, max_value_for_attr),#Esto es el máximo y mínimo valor por atributo, algo de numpy debe hacer esta parte
                loss=criterion,
                optimizer=optimizer,
                input_shape=columns,
                nb_classes=4,
            )
    
    classifier.fit(x_train, y_train, batch_size=64, nb_epochs=50)
    client_flex_model["model"] = classifier.model
    
    return client_flex_model

#selected_test_clients.map(train)

Se efectúa la agregación del modelo federado

In [6]:
from flex.pool import collect_client_diff_weights_pt
from flex.pool import fed_avg
from flex.pool import set_aggregated_diff_weights_pt


#pool.aggregators.map(collect_client_diff_weights_pt, selected_test_clients)
#pool.aggregators.map(fed_avg)
#pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)

Se define el ataque de inferencia de atributo, para ello se utiliza el framwork dicho previamente y se debe tener un registro de los clientes y rondas del modelo federado. En este caso, la inferencia tabaja sobre los presos de los modelos de los clientes situados en el agregador. Cada ataque se guarda en un registro.

In [7]:
from attack.my_models_attacks.tabular_atribute_inference import *
from flexclash.model import model_inference_known_agregator_information
import traceback

attack = tabular_inference_atributte(deepcopy(model_base))

round = 0
client_index = dict()

def find_client_id(index):
    for key, value in client_index.items():
        if value == index:
            return key
    return None


@model_inference_known_agregator_information
def inferencer(list_of_wigths_clients: list):

    x_train, y_train = transform_tabular(dataname="nursery", dataset = adv_data)
    min_value_for_attr = np.min(x_train, axis = 0)
    max_value_for_attr = np.max(x_train, axis = 0)
    columns = x_train[0].shape
    classes = len(set(y_train))

    for client in range(len(list_of_wigths_clients)):
        attack.update_model(list_of_wigths_clients[client])
        this_model = attack.model
        optimizer = optim.Adam(this_model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        client_id_act = find_client_id(client) 
        try:
            data_infer, model_infer_client, data_to_evaluate = attack.inf_atribute_attck_train(x_train, y_train, train_test_ratio = 0.5, model = this_model, criterion = criterion, 
                                                                    optim = optimizer, max_val = max_value_for_attr, min_val = min_value_for_attr, data_dimension = columns, 
                                                                    num_classes = classes)

            attack.save_summary_infer_for_clients(rounds = round, client_id = client_id_act, 
                                                infer = data_infer, model_to_future_infer = model_infer_client, 
                                                data_to_evaluate =data_to_evaluate)
        except Exception as e:
            print(f"Error: {e.__class__.__name__} - {e}")
            tb = traceback.format_exc()
            print(tb)


Se evalúa el modelo federado

In [8]:
def evaluate_global_model(server_flex_model: FlexModel, test_data: Dataset):#falta poner esto
    model = server_flex_model["model"]
    model.eval()
    test_loss = 0
    test_acc = 0
    total_count = 0
    model = model.to(device)
    criterion = server_flex_model["criterion"]
    # get test data as a torchvision object

    x_train, y_train = transform_tabular(dataname="nursery", dataset = test_data)
    test_dataset = transform_numpy_to_tensor_dataset(x_train, y_train)
    test_dataloader = DataLoader(
        test_dataset, batch_size=64, shuffle=True, pin_memory=False
    )
    losses = []
    with torch.no_grad():
        for data, target in tqdm(test_dataloader):
            total_count += target.size(0)
            data, target = data.to(device), target.to(device)
            output = model(data)
            losses.append(criterion(output, target).item())
            pred = output.data.max(1, keepdim=True)[1]
            test_acc += pred.eq(target.data.view_as(pred)).long().cpu().sum().item()

    test_loss = sum(losses) / len(losses)
    test_acc /= total_count
    return test_loss, test_acc

#metrics = pool.servers.map(evaluate_global_model)

#loss, acc = metrics[0]
#print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

Para limpiar los modelos en memoria. Opcional

In [9]:
def clean_up_models(client_model: FlexModel, _):
    import gc

    client_model.clear()
    gc.collect()

Se definen las rondas de entrenamiento del modelo federado. Para ello se mantiene un control de las rondas y clientes presentes en el modelo

In [10]:
def train_n_rounds(n_rounds, clients_per_round=20):
    pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func=build_server_model
    )
    global round
    global client_index
    
    for i in range(n_rounds):
        print(f"\nRunning round: {i+1} of {n_rounds}")
        selected_clients_pool = pool.clients.select(clients_per_round)
        selected_clients = selected_clients_pool.clients
        pool.servers.map(deploy_serv, selected_clients)
        selected_clients.map(train)
        pool.aggregators.map(collect_client_diff_weights_pt, selected_clients)
        if i != 0:
            pos_c_list=0
            client_index = dict()
            for i in selected_clients.actor_ids:
                print("Cliente id:", i)
                client_index[i] = pos_c_list
                pos_c_list+=1
            pool.aggregators.map(inferencer)
        pool.aggregators.map(fed_avg)
        pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)
        metrics = pool.servers.map(evaluate_global_model)
        selected_clients.map(clean_up_models)
        loss, acc = metrics[0]
        print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

In [11]:
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("default")
import logging
logging.captureWarnings(True)
logging.getLogger("py.warnings").setLevel(logging.ERROR)

train_n_rounds(4, clients_per_round=2)


Running round: 1 of 4
Después de la modif antes de fedavg tensor(-2.8831)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

100%|██████████| 21/21 [00:00<00:00, 1861.46it/s]


KeyboardInterrupt: 